# 🔧 Notebook 2 — Read-Repair: Heal Replicas as a Side Effect of Reads

**Read-repair** is a classic trick from Dynamo-style databases (Cassandra, DynamoDB, Riak):

> On every read, the coordinator asks **several** replicas for the same key. If it notices one of them returned a stale value, it **writes the fresh value back to the stale replica** — right there, during the read.

Convergence happens **lazily, paid for by reads**, instead of via a dedicated background scan.

In this notebook we build read-repair step by step (bad → better → best) and end with a side-by-side comparison vs. anti-entropy.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/read-repair
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🧱 A safer replica: last-write-wins on writes

Before building the coordinator, tighten the replica: a replica should **never go backwards**. If it already holds a newer value, ignore the older write. This is **last-write-wins (LWW)** applied on every write.


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional
import random, time

Entry = Tuple[str, int]

@dataclass
class Replica:
    name: str
    data: Dict[str, Entry] = field(default_factory=dict)
    repairs_received: int = 0   # for stats

    def write(self, k: str, v: str, ts: int, *, is_repair: bool = False) -> bool:
        cur = self.data.get(k)
        if cur is None or ts > cur[1]:
            self.data[k] = (v, ts)
            if is_repair:
                self.repairs_received += 1
            return True
        return False

    def read(self, k: str) -> Optional[Entry]:
        return self.data.get(k)

def fresh_cluster():
    r1, r2, r3 = Replica("r1"), Replica("r2"), Replica("r3")
    r1.write("user:42", "Alice v2", 200)
    r2.write("user:42", "Alice v2", 200)
    r3.write("user:42", "Alice v1", 100)   # r3 is behind
    return [r1, r2, r3]


## 🟥 BAD: read-one, never repair (from Notebook 1)

We keep this as the baseline: stale reads, no healing.


In [ ]:
class ReadOneCoordinator:
    def __init__(self, replicas): self.replicas = replicas
    def read(self, k):
        r = random.choice(self.replicas)
        return r.name, r.read(k)

random.seed(0)
cluster = fresh_cluster()
co = ReadOneCoordinator(cluster)
for _ in range(5):
    print(co.read("user:42"))

print("\nr3 after 5 reads:", cluster[2].read("user:42"), "  <- still stale")


## 🟨 BETTER: quorum read — pick the freshest of R replicas

Ask `R` replicas, keep the one with the highest timestamp. This **fixes the client-visible staleness** but the stale replica is still stale on disk — next time we don't include it in the quorum, the same issue reappears.


In [ ]:
class QuorumReadCoordinator:
    def __init__(self, replicas): self.replicas = replicas

    def read(self, k, R=2):
        # Ask the first R replicas. In reality you would ask all N and wait for R.
        chosen = self.replicas[:R]
        responses = [(r, r.read(k)) for r in chosen]
        valid = [(r, val) for r, val in responses if val is not None]
        if not valid: return None
        winner_r, (val, ts) = max(valid, key=lambda x: x[1][1])
        return val, ts

random.seed(0)
cluster = fresh_cluster()
co = QuorumReadCoordinator(cluster)
print("quorum(r1,r2):", co.read("user:42"))         # fresh — both had v2
# If the quorum happens to include r3 but not both fresh replicas, we can still win
# as long as one fresh replica is in the quorum.
print("r3 on disk :", cluster[2].read("user:42"))   # still stale!


## 🟩 BEST: read-repair — on every read, push the winner to stragglers

Do the quorum read, then for any replica whose value is older (or missing), **write the winner back** with the winning timestamp. Because replicas use LWW, repair writes are idempotent and safe to retry.


In [ ]:
class ReadRepairCoordinator:
    def __init__(self, replicas: List[Replica]):
        self.replicas = replicas
        self.repairs_triggered = 0

    def read(self, k: str, R: int = 2):
        responses = [(r, r.read(k)) for r in self.replicas[:R]]
        valid = [(r, val) for r, val in responses if val is not None]
        if not valid:
            return None
        _, (win_val, win_ts) = max(valid, key=lambda x: x[1][1])

        # Repair EVERY replica that is behind (including ones we didn't query).
        for r in self.replicas:
            cur = r.read(k)
            if cur is None or cur[1] < win_ts:
                print(f"  🔧 repairing {r.name}: {cur} -> ({win_val!r}, {win_ts})")
                r.write(k, win_val, win_ts, is_repair=True)
                self.repairs_triggered += 1
        return win_val, win_ts

random.seed(0)
cluster = fresh_cluster()
co = ReadRepairCoordinator(cluster)

print("read 1:", co.read("user:42"))
print("read 2:", co.read("user:42"), "  <- nothing left to repair")
print("r3 on disk now:", cluster[2].read("user:42"))
print("total repairs:", co.repairs_triggered)


## ⏱️ Blocking vs. asynchronous repair

Above we repaired **inside** the client read path — the client waits for the repair write before seeing a response. That guarantees the stale replica is fixed *before* we return, but it raises tail latency.

A common optimization:

- **Blocking repair** — repair synchronously only when the freshest replica disagrees with the quorum majority. Used sparingly.
- **Asynchronous repair** — return the fresh value to the client immediately, then repair stragglers in a background task.

Below we simulate async repair with a simple queue.


In [ ]:
from collections import deque

class AsyncReadRepairCoordinator:
    def __init__(self, replicas: List[Replica]):
        self.replicas = replicas
        self.repair_queue: deque = deque()

    def read(self, k: str, R: int = 2):
        responses = [(r, r.read(k)) for r in self.replicas[:R]]
        valid = [(r, val) for r, val in responses if val is not None]
        if not valid: return None
        _, (win_val, win_ts) = max(valid, key=lambda x: x[1][1])
        for r in self.replicas:
            cur = r.read(k)
            if cur is None or cur[1] < win_ts:
                # Enqueue instead of writing now — client doesn't wait.
                self.repair_queue.append((r, k, win_val, win_ts))
        return win_val, win_ts      # returned immediately

    def drain_repairs(self):
        while self.repair_queue:
            r, k, v, ts = self.repair_queue.popleft()
            r.write(k, v, ts, is_repair=True)

cluster = fresh_cluster()
co = AsyncReadRepairCoordinator(cluster)
print("client sees:", co.read("user:42"))
print("r3 BEFORE drain:", cluster[2].read("user:42"))  # still stale momentarily
co.drain_repairs()                                     # the background worker runs
print("r3 AFTER drain :", cluster[2].read("user:42"))


**Trade-off:** async repair keeps read latency low, but there is a short window where a *different* coordinator reading the same key could still pick up a stale value. That's the "eventual" in eventual consistency.


## 🎲 Probabilistic read-repair (a real knob from Cassandra)

Cassandra used to expose `read_repair_chance` / `dclocal_read_repair_chance`: the probability, per read, that the coordinator queries **all** replicas (not just the quorum) and runs repair. With probability `1 - p` the read is cheap and skips repair.

Why? Reading all replicas on every request costs bandwidth. If your workload is read-heavy, most keys get repaired quickly even at `p = 0.1`.

Below we simulate the convergence rate for a few values of `p`.


In [ ]:
def simulate(p_repair: float, num_keys: int = 200, num_reads: int = 2000, seed: int = 0):
    random.seed(seed)
    # 3 replicas, r3 is missing every initial write
    reps = [Replica(f"r{i}") for i in range(3)]
    for k in range(num_keys):
        reps[0].write(f"k{k}", "v-new", 200)
        reps[1].write(f"k{k}", "v-new", 200)
        reps[2].write(f"k{k}", "v-old", 100)

    for _ in range(num_reads):
        k = f"k{random.randrange(num_keys)}"
        if random.random() < p_repair:
            # full repair path — ask everyone
            answers = [(r, r.read(k)) for r in reps]
            _, (val, ts) = max(answers, key=lambda x: x[1][1])
            for r in reps:
                cur = r.read(k)
                if cur is None or cur[1] < ts:
                    r.write(k, val, ts, is_repair=True)
        else:
            # cheap path — one replica, no repair
            random.choice(reps).read(k)

    still_stale = sum(1 for k in range(num_keys) if reps[2].data[f"k{k}"][1] < 200)
    return still_stale

for p in (0.0, 0.05, 0.2, 0.5, 1.0):
    stale = simulate(p)
    print(f"p_repair={p:<4}  keys still stale on r3: {stale:>3}/200")


Notice how even small probabilities knock the stale count down fast — because each read is another chance to repair one of the 200 keys.


## 🧩 Gotcha: LWW is simple but lossy

Read-repair needs a rule to decide the "winner". LWW (highest timestamp) is the easiest, but if two clients write **concurrently**, LWW silently drops one of them. Production systems use richer schemes:

- **Vector clocks** (Riak, early Dynamo): detect concurrent writes and return *both* to the client as **siblings** — let the application merge.
- **CRDTs**: data types (counters, sets) whose merge function is built in and commutative.
- **Hybrid logical clocks (HLC)**: timestamps that are both wall-clock-ish and causality-aware.

For this lab we stay with LWW; just know it's not the end of the story.
